# Block 3 Solution — Validate and Check

Completed version of `03_validate_check/validate_check_exercise.ipynb`. The markdown notes at
each TODO explain the approach.

In [1]:
import os
import odmlib.define_loader as DL
import odmlib.loader as LD
from odmlib.odm_parser import ODMSchemaValidator
from odmlib import create_oid_checker
from odmlib.define_2_1.rules.metadata_schema import MetadataSchema

os.makedirs("output", exist_ok=True)

def load_define(path):
    """Load a Define-XML v2.1 file and return the ODM root object."""
    loader = LD.ODMLoader(DL.XMLDefineLoader(model_package="define_2_1"))
    loader.open_odm_document(path)
    return loader.root()      # one root() call - keep this reference for any edits

xsd = ODMSchemaValidator(standard="define", version="2.1")
print("ready")

ready


In [2]:
xsd.validate_file("../data/defineV21-SDTM.xml")
print("layer 1 (XSD):         PASS")

layer 1 (XSD):         PASS


In [3]:
odm = load_define("../data/defineV21-SDTM.xml")

odm.verify_oids(create_oid_checker("define_2_1"))
print("layer 2 (OID ref/def): PASS")

odm.verify_conformance(MetadataSchema())
print("layer 3 (conformance): PASS")

odm.verify_order()
print("layer 4 (order):       PASS")

layer 2 (OID ref/def): PASS


layer 3 (conformance): PASS
layer 4 (order):       PASS


In [4]:
errors = odm.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
print(f"combined validate: {len(errors)} errors")

combined validate: 0 errors


## TODO 1 — Diagnose a schema-invalid file

`iter_errors` enumerates every schema violation instead of raising on the first. **Answer:**
the required `def:Context` attribute is missing from the root `ODM` element — the same
attribute you set when constructing `DEF.ODM(...)` in Block 2.

In [5]:
for err in xsd.xsd.iter_errors("../data/defineV21-SDTM-invalid.xml"):
    print("reason:", err.reason)
    print("path:  ", err.path)

reason: missing required attribute '{http://www.cdisc.org/ns/def/v2.1}Context'
path:   /ODM


## TODO 2 — Find the reference errors

The file is schema-valid (to XSD an OID is just a string) but layer 2 walks every reference.
**Answer:** an `ItemRef` points at `IT.DM.SEXX`, which is never defined.

In [6]:
from odmlib.exceptions import OdmlibOIDError

broken = load_define("../data/define_broken_refs.xml")
mdv = broken.Study.MetaDataVersion

try:
    broken.verify_oids(create_oid_checker("define_2_1"))
    print("no reference errors")
except OdmlibOIDError as e:
    print("reference error found:")
    print(e)

reference error found:
OID IT.DM.SEXX referenced in attribute ItemOID is not found.
  Hint: Define an element with OID 'IT.DM.SEXX' before referencing it via ItemOID.


## TODO 3 — Repair it until it validates clean

Everything happens on the object tree — no text editing. Note the order: fix dangling refs
first (`unreferenced_oids` raises while any remain), then deal with orphans. Every check gets
a **fresh** OID checker.

In [7]:
# step 1: fix the ItemRef typo
bad_ref = next(r for r in mdv.ItemGroupDef[0].ItemRef if r.ItemOID == "IT.DM.SEXX")
bad_ref.ItemOID = "IT.DM.SEX"

# step 2: references are clean now
broken.verify_oids(create_oid_checker("define_2_1"))
print("verify_oids: PASS")

verify_oids: PASS


In [8]:
# step 3: with refs clean, hunt orphans
orphans = broken.unreferenced_oids(create_oid_checker("define_2_1"))
print("orphans:", orphans)

for oid in orphans:
    orphan_item = mdv.find("ItemDef", "OID", oid)
    mdv.ItemDef.remove(orphan_item)
    print("removed", oid)

orphans: {'IT.DM.ARMCD': 'ItemOID'}
removed IT.DM.ARMCD


In [9]:
# steps 4-5: combined validate to empty, write, XSD-validate the repaired file
errors = broken.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
print(f"combined validate: {len(errors)} errors")

broken.write_xml("output/define_repaired.xml")
xsd.validate_file("output/define_repaired.xml")
print("wrote and XSD-validated output/define_repaired.xml")

combined validate: 0 errors
wrote and XSD-validated output/define_repaired.xml


## TODO 4 — Why you need both the object checks and XSD

**Answer: only XSD catches it.** `def:Class/@Name` is constrained by an XSD enumeration; the
object-level layers have nothing to say about this attribute's value. The reverse was true in
TODO 2 — the dangling OID passed XSD and only layer 2 caught it. Different layers, different
failure classes: run both.

In [10]:
bad_class = load_define("../data/defineV21-SDTM-invalid-class.xml")

errors = bad_class.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
print(f"object layers (combined validate): {len(errors)} errors")

print("\nXSD:")
for err in xsd.xsd.iter_errors("../data/defineV21-SDTM-invalid-class.xml"):
    print("reason:", err.reason[:120])

object layers (combined validate): 0 errors

XSD:


reason: attribute Name='NONSENSE': value must be one of ['ADAM OTHER', 'BASIC DATA STRUCTURE', 'DEVICE LEVEL ANALYSIS DATASET', 


## Stretch — permissive-mode repair

Strict mode refuses the file at construction time (missing required `Repeating`). Permissive
mode gets it into objects; `validate(collect_errors=True, ...)` lists everything wrong in one
pass; the fixes are ordinary attribute assignments.

In [11]:
import odmlib
from odmlib.exceptions import OdmlibRequiredAttributeError

try:
    load_define("../data/nonconformant_define21.xml")
except OdmlibRequiredAttributeError as e:
    print("strict load fails:", e)

with odmlib.permissive():
    nc = load_define("../data/nonconformant_define21.xml")
print("\npermissive load OK")

errors = nc.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
for err in errors:
    print(type(err).__name__, "-", str(err).splitlines()[0])

strict load fails: Missing required keyword argument Repeating in ItemGroupDef
  Hint: Attribute 'Repeating' is required when constructing ItemGroupDef

permissive load OK
OdmlibConformanceError - Study.MetaDataVersion.ItemDef.0.Origin.0.Type: unallowed value Bogus
OdmlibConformanceError - Study.MetaDataVersion.ItemGroupDef.0.Repeating: required field


In [12]:
nc_mdv = nc.Study.MetaDataVersion

# fix 1: the missing required attribute
nc_mdv.ItemGroupDef[0].Repeating = "No"

# fix 2: the invalid Origin type
nc_mdv.ItemDef[0].Origin[0].Type = "Collected"

errors = nc.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
print(f"after repair: {len(errors)} errors")

after repair: 0 errors


Load (permissively if needed) → collect every error → fix objects → re-validate to clean →
write. That loop is the core odmlib workflow — and with ~20 lines of code you've built the
essence of a Define-XML repair tool.